# Tool Calling

In [1]:
import dotenv
from agents import Agent, ModelSettings, Runner, function_tool, trace

dotenv.load_dotenv()

True

Create a static calorie table that we can use as a tool:

create a tool tha takes a food item and returns the calories of that food item

In [ ]:
@function_tool
def get_food_calories(food_item: str) -> str:
    # the agent will use the docstring as the tool description to understand what it does, which parameters it takes and what it returns.
    # so always give a verbose docstring
    """
    Get calorie information for common foods to help with nutrition tracking.

    Args:
        food_item: Name of the food (e.g., "apple", "banana")

    Returns:
        Calorie information per standard serving
    """
    # Dictionary: Simple calorie database - in real world, you'd use USDA API
    calorie_data = {
        "apple": "80 calories per medium apple (182g)",
        "banana": "105 calories per medium banana (118g)",
        "broccoli": "25 calories per 1 cup chopped (91g)",
        "almonds": "164 calories per 1oz (28g) or about 23 nuts",
    }

    # once we find the food (that we turned lower case for matching), return the calorie info
    # If we cannot find the food item, then we return a default message

    food_key = food_item.lower()
    if food_key in calorie_data:
        return f"{food_item.title()}: {calorie_data[food_key]}"
    else:
        return f"I don't have calorie data for {food_item} in my database. Try common foods like apple, chicken breast, or rice."

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `get_food_calories` function_

In [3]:
get_food_calories('banana')

'Banana: 105 calories per medium banana (118g)'

Let's test a negative case as well

In [4]:
get_food_calories('grapes')

"I don't have calorie data for grapes in my database. Try common foods like apple, chicken breast, or rice."

In [11]:
# the agent definition fro mthe last exercise. Let's make it call the tool
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    You MUST use first the tool 'get_food_calories' to get calorie information about foods.
    If the tool doesn't have the information, you can come up with your own response but you need to explicitly state is that the tools didn't have the information.
    At the end don't ask back, just give the final answer.
    """,
    tools=[get_food_calories])

In [12]:
with trace("Nutrition Assistant with tools"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple and grapes?"
    )
    print(result.final_output)

Banana: 105 cal
Apple: 80 cal
Grapes: not in tool data; approximate 104 cal per cup (151 g)

Total ≈ 289 calories


Enforce tools use:

In [ ]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    """,
    tools=[get_food_calories],
    model_settings=ModelSettings(tool_choice="get_food_calories"),
)

with trace("Nutrition Assistant with tools enforced"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple?"
    )
    print(result.final_output)